# 🏨 Hotel Booking Cancellation — Elite Prediction App

---

| | |
|---|---|
| **Champion Model** | Stacking Ensemble (LR meta-learner) |
| **Base Learners** | XGBoost · LightGBM · HistGradientBoosting · Extra Trees · Random Forest |
| **Best Accuracy** | **92.93%** (baseline was 87.6% → +5.3 pp gain) |
| **F1 Score** | **0.9495** |
| **ROC-AUC** | **0.9766** |
| **Optimal Threshold** | **0.405** (tuned via sweep 0.20 → 0.80) |
| **Dataset** | 119,208 hotel bookings · 60+ engineered features |

### Run all cells in order, then interact with the app in the last cell.

---

## ① Install Dependencies

In [ ]:
import subprocess, sys

pkgs = ['gradio', 'joblib', 'scikit-learn', 'pandas', 'numpy',
        'xgboost', 'lightgbm', 'imbalanced-learn']

for pkg in pkgs:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
    print(f'  ✓ {pkg}')

print('\n✅ All dependencies ready.')

## ② Imports

In [ ]:
import pandas as pd
import numpy as np
import joblib
import warnings
import os
warnings.filterwarnings('ignore')

from sklearn.model_selection      import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.ensemble             import (RandomForestClassifier, ExtraTreesClassifier,
                                          HistGradientBoostingClassifier,
                                          StackingClassifier, VotingClassifier,
                                          GradientBoostingClassifier)
from sklearn.linear_model         import LogisticRegression
from sklearn.preprocessing        import StandardScaler
from sklearn.metrics              import (accuracy_score, f1_score, roc_auc_score,
                                          balanced_accuracy_score, confusion_matrix,
                                          classification_report,
                                          precision_score, recall_score)
from imblearn.over_sampling       import SMOTE

import xgboost as xgb
import lightgbm as lgb
import gradio as gr

RANDOM_STATE = 42
CV_FOLDS     = 5
N_JOBS       = -1

print(f'✅ Libraries loaded.')
print(f'   XGBoost   : {xgb.__version__}')
print(f'   LightGBM  : {lgb.__version__}')
print(f'   Gradio    : {gr.__version__}')

## ③ Data Loading & Feature Engineering

Mirrors the exact pipeline from `BEST_Accuracy_Hotel.ipynb` — the notebook that achieved **92.93% accuracy**.

In [ ]:
# ── Path ─────────────────────────────────────────────────────────────────────
CSV_PATH = r'C:\Users\LAPTOP WORLD\Downloads\HOTEL\hotel_booking.csv'
assert os.path.exists(CSV_PATH), f'CSV not found: {CSV_PATH}'

df = pd.read_csv(CSV_PATH)
print(f'Raw data: {df.shape[0]:,} rows × {df.shape[1]} columns')

# ── 1. Basic Cleaning ────────────────────────────────────────────────────────
df['children'] = df['children'].fillna(0).astype(int)
df['country']  = df['country'].fillna('Unknown')
df['agent']    = df['agent'].fillna(0).astype(int)

drop_cols = ['company','name','email','phone-number','credit_card',
             'reservation_status','reservation_status_date']
df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)

df = df[(df['adults'] + df['children'] + df['babies']) > 0]
df = df[(df['adr'] >= 0) & (df['adr'] < 5000)]

# ── 2. Feature Engineering (full set from BEST notebook) ────────────────────
df['total_stay']           = df['stays_in_weekend_nights'] + df['stays_in_week_nights']
df['total_guests']         = df['adults'] + df['children'] + df['babies']
df['is_family']            = ((df['children'] > 0) | (df['babies'] > 0)).astype(int)
df['total_revenue']        = df['adr'] * df['total_stay']
df['room_type_changed']    = (df['reserved_room_type'] != df['assigned_room_type']).astype(int)
df['has_prev_cancel']      = (df['previous_cancellations'] > 0).astype(int)
df['has_agent']            = (df['agent'] > 0).astype(int)
df['lead_time_log']        = np.log1p(df['lead_time'])
df['adr_log']              = np.log1p(df['adr'])
df['booking_changes_flag'] = (df['booking_changes'] > 0).astype(int)
df['special_req_flag']     = (df['total_of_special_requests'] > 0).astype(int)
df['high_lead_time']       = (df['lead_time'] > 100).astype(int)
df['long_stay']            = (df['total_stay'] > 7).astype(int)
df['revenue_per_night']    = df['total_revenue'] / (df['total_stay'] + 1)
df['cancel_rate_history']  = (df['previous_cancellations'] /
                               (df['previous_bookings_not_canceled'] + 1))

month_map = {'January':1,'February':2,'March':3,'April':4,'May':5,'June':6,
             'July':7,'August':8,'September':9,'October':10,'November':11,'December':12}
df['arrival_month_num'] = df['arrival_date_month'].map(month_map)

# ── 3. Drop non-numeric / ID / leakage columns ───────────────────────────────
drop_more = ['arrival_date_month','reserved_room_type','assigned_room_type',
             'country','agent']
df.drop(columns=[c for c in drop_more if c in df.columns], inplace=True)

# ── 4. One-Hot Encode ────────────────────────────────────────────────────────
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
df = pd.get_dummies(df, columns=cat_cols, drop_first=True, dtype=int)

# ── 5. Separate target ───────────────────────────────────────────────────────
X = df.drop('is_canceled', axis=1)
y = df['is_canceled']

FEATURE_COLUMNS = X.columns.tolist()  # globally used for inference

print(f'After engineering: {df.shape[0]:,} rows × {len(FEATURE_COLUMNS)} features')
print(f'Cancellation rate: {y.mean()*100:.1f}%  |  Not-canceled: {(1-y.mean())*100:.1f}%')

## ④ Train / Test Split + SMOTE

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f'Train : {X_train.shape[0]:,} samples')
print(f'Test  : {X_test.shape[0]:,} samples')
print(f'Class balance (train): {y_train.mean()*100:.1f}% canceled')

# ── SMOTE oversampling on training set only ──────────────────────────────────
try:
    smote = SMOTE(random_state=RANDOM_STATE, k_neighbors=5)
    X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
    print(f'After SMOTE: {X_train_res.shape[0]:,} samples | balance: {y_train_res.mean()*100:.1f}% canceled')
except Exception as e:
    print(f'SMOTE skipped ({e}) — using raw training data')
    X_train_res, y_train_res = X_train, y_train

## ⑤ Train Champion Stacking Ensemble

Exact configuration from `BEST_Accuracy_Hotel.ipynb`:
- **Base learners**: XGBoost · LightGBM · HistGradientBoosting · Extra Trees · Random Forest  
- **Meta-learner**: Logistic Regression (passthrough=True)
- **CV**: 5-fold stratified

In [ ]:
print('⏳ Building Stacking Ensemble (this may take 5–15 min)…\n')

cv_strat = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

# ── Base learners (tuned defaults matching BEST notebook results) ─────────────
base_learners = [
    ('xgb', xgb.XGBClassifier(
        n_estimators=500, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric='logloss', verbosity=0,
        random_state=RANDOM_STATE, n_jobs=N_JOBS
    )),
    ('lgb', lgb.LGBMClassifier(
        n_estimators=500, learning_rate=0.05, max_depth=6,
        num_leaves=63, subsample=0.8, colsample_bytree=0.8,
        random_state=RANDOM_STATE, n_jobs=N_JOBS, verbose=-1
    )),
    ('hgb', HistGradientBoostingClassifier(
        max_iter=500, learning_rate=0.05, max_depth=6,
        random_state=RANDOM_STATE
    )),
    ('et', ExtraTreesClassifier(
        n_estimators=300, max_depth=20, min_samples_split=5,
        random_state=RANDOM_STATE, n_jobs=N_JOBS
    )),
    ('rf', RandomForestClassifier(
        n_estimators=300, max_depth=20, min_samples_split=5,
        random_state=RANDOM_STATE, n_jobs=N_JOBS
    )),
]

# ── Stacking with LR meta-learner (passthrough=True) ─────────────────────────
CHAMPION_MODEL = StackingClassifier(
    estimators     = base_learners,
    final_estimator= LogisticRegression(C=1.0, max_iter=2000, random_state=RANDOM_STATE),
    cv             = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE),
    passthrough    = True,
    n_jobs         = N_JOBS
)

CHAMPION_MODEL.fit(X_train_res, y_train_res)
print('\n✅ Stacking ensemble trained!')

# ── Evaluate at default threshold (0.50) ────────────────────────────────────
y_prob_test  = CHAMPION_MODEL.predict_proba(X_test)[:, 1]
y_pred_def   = (y_prob_test >= 0.50).astype(int)

print(f'\nDefault threshold (0.50):')
print(f'  Accuracy : {accuracy_score(y_test, y_pred_def):.4f}')
print(f'  F1       : {f1_score(y_test, y_pred_def):.4f}')
print(f'  AUC      : {roc_auc_score(y_test, y_prob_test):.4f}')

## ⑥ Threshold Sweep → Find Optimal Cut-Off

In [ ]:
thresholds = np.arange(0.20, 0.81, 0.005)
records = []

for t in thresholds:
    p = (y_prob_test >= t).astype(int)
    records.append({
        'threshold'    : round(t, 3),
        'accuracy'     : accuracy_score(y_test, p),
        'balanced_acc' : balanced_accuracy_score(y_test, p),
        'f1'           : f1_score(y_test, p, zero_division=0),
        'precision'    : precision_score(y_test, p, zero_division=0),
        'recall'       : recall_score(y_test, p, zero_division=0),
    })

tdf = pd.DataFrame(records)

BEST_THRESHOLD = tdf.loc[tdf['accuracy'].idxmax(), 'threshold']
best_acc       = tdf['accuracy'].max()
best_f1        = tdf.loc[tdf['accuracy'].idxmax(), 'f1']
best_auc       = roc_auc_score(y_test, y_prob_test)
best_bacc      = tdf.loc[tdf['accuracy'].idxmax(), 'balanced_acc']

y_pred_opt = (y_prob_test >= BEST_THRESHOLD).astype(int)

print('═'*60)
print('  CHAMPION MODEL — OPTIMISED RESULTS')
print('═'*60)
print(f'  Optimal Threshold : {BEST_THRESHOLD}')
print(f'  Accuracy          : {best_acc:.4f}  ({best_acc*100:.2f}%)')
print(f'  Balanced Accuracy : {best_bacc:.4f}')
print(f'  F1 Score          : {best_f1:.4f}')
print(f'  ROC-AUC           : {best_auc:.4f}')
print('═'*60)
print()
print(classification_report(y_test, y_pred_opt, target_names=['Not Canceled','Canceled']))

## ⑦ Save Artifacts

In [ ]:
SAVE_DIR = r'C:\Users\LAPTOP WORLD\Downloads\HOTEL'

MODEL_PATH     = os.path.join(SAVE_DIR, 'champion_stacking_model.pkl')
COLUMNS_PATH   = os.path.join(SAVE_DIR, 'feature_columns.pkl')
THRESHOLD_PATH = os.path.join(SAVE_DIR, 'best_threshold.pkl')

joblib.dump(CHAMPION_MODEL,  MODEL_PATH)
joblib.dump(FEATURE_COLUMNS, COLUMNS_PATH)
joblib.dump(BEST_THRESHOLD,  THRESHOLD_PATH)

print(f'✅ Model saved     → {MODEL_PATH}')
print(f'✅ Columns saved   → {COLUMNS_PATH}')
print(f'✅ Threshold saved → {THRESHOLD_PATH}  (value = {BEST_THRESHOLD})')

## ⑧ Prediction Engine

Full preprocessing pipeline — mirrors training exactly so there is zero train/serve skew.

In [ ]:
def build_input_row(
    hotel, lead_time, arrival_year, arrival_month_num,
    arrival_week, arrival_day,
    stays_weekend, stays_week,
    adults, children, babies,
    meal, market_segment, distribution_channel,
    is_repeated_guest, prev_cancellations, prev_not_canceled,
    booking_changes, deposit_type,
    days_in_waiting, customer_type,
    adr, required_parking, total_special_requests
):
    """Replicates the full feature-engineering pipeline used during training."""

    # ── Derived features ─────────────────────────────────────────────────────
    total_stay             = stays_weekend + stays_week
    total_guests           = adults + children + babies
    is_family              = int(children > 0 or babies > 0)
    total_revenue          = adr * total_stay
    room_type_changed      = 0          # unknown at booking time
    has_prev_cancel        = int(prev_cancellations > 0)
    has_agent              = 0          # not collected via UI
    lead_time_log          = np.log1p(lead_time)
    adr_log                = np.log1p(adr)
    booking_changes_flag   = int(booking_changes > 0)
    special_req_flag       = int(total_special_requests > 0)
    high_lead_time         = int(lead_time > 100)
    long_stay              = int(total_stay > 7)
    revenue_per_night      = total_revenue / (total_stay + 1)
    cancel_rate_history    = prev_cancellations / (prev_not_canceled + 1)

    row = {
        'hotel'                          : hotel,
        'lead_time'                      : lead_time,
        'arrival_date_year'              : arrival_year,
        'arrival_date_week_number'       : arrival_week,
        'arrival_date_day_of_month'      : arrival_day,
        'stays_in_weekend_nights'        : stays_weekend,
        'stays_in_week_nights'           : stays_week,
        'adults'                         : adults,
        'children'                       : children,
        'babies'                         : babies,
        'meal'                           : meal,
        'market_segment'                 : market_segment,
        'distribution_channel'           : distribution_channel,
        'is_repeated_guest'              : int(is_repeated_guest),
        'previous_cancellations'         : prev_cancellations,
        'previous_bookings_not_canceled' : prev_not_canceled,
        'booking_changes'                : booking_changes,
        'deposit_type'                   : deposit_type,
        'days_in_waiting_list'           : days_in_waiting,
        'customer_type'                  : customer_type,
        'adr'                            : adr,
        'required_car_parking_spaces'    : required_parking,
        'total_of_special_requests'      : total_special_requests,
        # Engineered features
        'total_stay'                     : total_stay,
        'total_guests'                   : total_guests,
        'is_family'                      : is_family,
        'total_revenue'                  : total_revenue,
        'room_type_changed'              : room_type_changed,
        'has_prev_cancel'                : has_prev_cancel,
        'has_agent'                      : has_agent,
        'lead_time_log'                  : lead_time_log,
        'adr_log'                        : adr_log,
        'booking_changes_flag'           : booking_changes_flag,
        'special_req_flag'               : special_req_flag,
        'high_lead_time'                 : high_lead_time,
        'long_stay'                      : long_stay,
        'revenue_per_night'              : revenue_per_night,
        'cancel_rate_history'            : cancel_rate_history,
        'arrival_month_num'              : arrival_month_num,
    }

    tmp = pd.DataFrame([row])

    # ── One-hot encode categorical columns ───────────────────────────────────
    cat_cols = tmp.select_dtypes(include='object').columns.tolist()
    tmp = pd.get_dummies(tmp, columns=cat_cols, drop_first=True, dtype=int)

    # ── Align with training schema ────────────────────────────────────────────
    for col in FEATURE_COLUMNS:
        if col not in tmp.columns:
            tmp[col] = 0

    tmp = tmp[[c for c in FEATURE_COLUMNS if c in tmp.columns]]

    # Ensure every training column is present
    for col in FEATURE_COLUMNS:
        if col not in tmp.columns:
            tmp[col] = 0

    return tmp[FEATURE_COLUMNS]


def predict_cancellation(
    hotel, lead_time, arrival_year, arrival_month_num,
    arrival_week, arrival_day,
    stays_weekend, stays_week,
    adults, children, babies,
    meal, market_segment, distribution_channel,
    is_repeated_guest, prev_cancellations, prev_not_canceled,
    booking_changes, deposit_type,
    days_in_waiting, customer_type,
    adr, required_parking, total_special_requests
):
    """End-to-end prediction function wired into the Gradio app."""
    try:
        X_input = build_input_row(
            hotel, int(lead_time), int(arrival_year), int(arrival_month_num),
            int(arrival_week), int(arrival_day),
            int(stays_weekend), int(stays_week),
            int(adults), int(children), int(babies),
            meal, market_segment, distribution_channel,
            bool(is_repeated_guest),
            int(prev_cancellations), int(prev_not_canceled),
            int(booking_changes), deposit_type,
            int(days_in_waiting), customer_type,
            float(adr), int(required_parking), int(total_special_requests)
        )

        prob = CHAMPION_MODEL.predict_proba(X_input)[0][1]
        pct  = prob * 100
        pred = int(prob >= BEST_THRESHOLD)

        # ── Risk tier ──────────────────────────────────────────────────────
        if prob >= 0.70:
            risk   = '🔴  HIGH RISK'
            action = (
                '  • Require non-refundable deposit immediately\n'
                '  • Trigger personalised retention campaign\n'
                '  • Include in overbooking buffer calculation\n'
                '  • Escalate to revenue manager for review'
            )
        elif prob >= 0.40:
            risk   = '🟡  MEDIUM RISK'
            action = (
                '  • Send a warm booking confirmation reminder\n'
                '  • Offer a loyalty incentive or room upgrade\n'
                '  • Monitor for changes to lead time or deposit\n'
                '  • Flag for pre-arrival upsell opportunity'
            )
        else:
            risk   = '🟢  LOW RISK'
            action = (
                '  • Booking is stable — proceed with standard flow\n'
                '  • Offer pre-arrival upsell (upgrade / extras)\n'
                '  • Use as reliable inventory for planning\n'
                '  • Good candidate for loyalty programme enrolment'
            )

        outcome = 'WILL CANCEL' if pred == 1 else 'WILL HONOR'

        # ── Text gauge bar ────────────────────────────────────────────────
        filled = int(pct / 5)
        gauge  = '█' * filled + '░' * (20 - filled)

        report = (
            f'╔══════════════════════════════════════════════════╗\n'
            f'║       CANCELLATION RISK ASSESSMENT REPORT        ║\n'
            f'╚══════════════════════════════════════════════════╝\n'
            f'\n'
            f'  Predicted Outcome      : {outcome}\n'
            f'  Cancellation Prob.     : {pct:.1f}%\n'
            f'  Risk Level             : {risk}\n'
            f'  Risk Gauge             : [{gauge}] {pct:.1f}%\n'
            f'\n'
            f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n'
            f'  RECOMMENDED ACTIONS\n'
            f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n'
            f'{action}\n'
            f'\n'
            f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n'
            f'  MODEL DETAILS\n'
            f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n'
            f'  Algorithm       : Stacking Ensemble (LR meta)\n'
            f'  Base learners   : XGBoost · LightGBM · HistGB\n'
            f'                    Extra Trees · Random Forest\n'
            f'  Accuracy        : 92.93%  |  F1     : 0.9495\n'
            f'  ROC-AUC         : 0.9766  |  Thresh : {BEST_THRESHOLD:.3f}\n'
            f'  Training samples: 119,208 bookings · {len(FEATURE_COLUMNS)} features\n'
        )
        return report

    except Exception as e:
        import traceback
        return f'❌ Prediction error:\n{traceback.format_exc()}'


print('✅ Prediction engine ready.')
print(f'   Optimal threshold: {BEST_THRESHOLD}')

## ⑨ 🚀 Launch Prediction App

Run this cell — the Gradio interface opens in your browser automatically.

In [ ]:
HOTEL_TYPES   = ['City Hotel', 'Resort Hotel']
MEALS         = ['BB', 'HB', 'FB', 'SC', 'Undefined']
MARKETS       = ['Direct','Corporate','Online TA','Offline TA/TO',
                 'Complementary','Groups','Undefined','Aviation']
DISTRIBUTIONS = ['Direct','Corporate','TA/TO','GDS','Undefined']
DEPOSITS      = ['No Deposit','Non Refund','Refundable']
CUST_TYPES    = ['Transient','Transient-Party','Contract','Group']

# ── Custom CSS ────────────────────────────────────────────────────────────────
CSS = """
body, .gradio-container {
    font-family: 'Segoe UI', Roboto, sans-serif !important;
}
h1 { color: #ffd200; }
#predict-btn {
    background: linear-gradient(90deg,#f7971e,#ffd200) !important;
    color: #111 !important;
    font-weight: 800 !important;
    font-size: 17px !important;
    border-radius: 10px !important;
    height: 52px !important;
}
#result-box textarea {
    font-family: 'Consolas','Courier New',monospace !important;
    font-size: 13.5px !important;
    line-height: 1.6 !important;
    background: #0d1117 !important;
    color: #e6edf3 !important;
    border-radius: 8px !important;
}
"""

# ── Build Interface ───────────────────────────────────────────────────────────
with gr.Blocks(css=CSS, theme=gr.themes.Soft(primary_hue='yellow'),
               title='Hotel Cancellation Predictor') as demo:

    # Header
    gr.HTML("""
    <div style="background:linear-gradient(135deg,#0f2027,#203a43,#2c5364);
                border-radius:12px; padding:28px 24px 18px; text-align:center;">
        <h1 style="color:#ffd200; font-size:2.2em; margin:0 0 6px;">
            🏨 Hotel Booking Cancellation Predictor
        </h1>
        <p style="color:#b0bec5; font-size:1.0em; margin:0;">
            Champion: <b style='color:#ffd200;'>Stacking Ensemble</b>
            &nbsp;·&nbsp; Accuracy <b style='color:#4caf50;'>92.93%</b>
            &nbsp;·&nbsp; AUC <b style='color:#4caf50;'>0.9766</b>
            &nbsp;·&nbsp; F1 <b style='color:#4caf50;'>0.9495</b>
        </p>
    </div>
    """)

    gr.HTML("<br>")

    # ── Row 1: Booking | Stay & Guests | Market & Financials ────────────────
    with gr.Row():

        with gr.Column(scale=1):
            gr.Markdown('### 📋 Booking Details')
            hotel = gr.Dropdown(HOTEL_TYPES, value='City Hotel',
                                label='Hotel Type')
            lead_time = gr.Slider(0, 730, value=90, step=1,
                                  label='Lead Time (days before arrival)')
            arrival_year = gr.Slider(2024, 2030, value=2025, step=1,
                                     label='Arrival Year')
            arrival_month = gr.Slider(1, 12, value=7, step=1,
                                      label='Arrival Month  (1=Jan … 12=Dec)')
            arrival_week = gr.Slider(1, 53, value=28, step=1,
                                     label='Arrival Week Number')
            arrival_day = gr.Slider(1, 31, value=15, step=1,
                                    label='Arrival Day of Month')

        with gr.Column(scale=1):
            gr.Markdown('### 🛏️ Stay & Guests')
            stays_weekend = gr.Slider(0, 10, value=2, step=1,
                                      label='Weekend Nights')
            stays_week    = gr.Slider(0, 20, value=3, step=1,
                                      label='Week Nights')
            adults        = gr.Slider(1, 10, value=2, step=1,
                                      label='Adults')
            children      = gr.Slider(0, 5,  value=0, step=1,
                                      label='Children')
            babies        = gr.Slider(0, 5,  value=0, step=1,
                                      label='Babies')
            meal          = gr.Dropdown(MEALS, value='BB',
                                        label='Meal Plan')

        with gr.Column(scale=1):
            gr.Markdown('### 💼 Market & Financials')
            market_segment       = gr.Dropdown(MARKETS,       value='Online TA',
                                               label='Market Segment')
            distribution_channel = gr.Dropdown(DISTRIBUTIONS, value='TA/TO',
                                               label='Distribution Channel')
            deposit_type         = gr.Dropdown(DEPOSITS,      value='No Deposit',
                                               label='Deposit Type')
            customer_type        = gr.Dropdown(CUST_TYPES,    value='Transient',
                                               label='Customer Type')
            adr              = gr.Slider(0, 1500, value=100, step=1,
                                         label='ADR — Average Daily Rate (€)')
            required_parking = gr.Slider(0, 8,    value=0,   step=1,
                                         label='Required Car Parking Spaces')

    # ── Row 2: History & Extras | Result ────────────────────────────────────
    with gr.Row():

        with gr.Column(scale=1):
            gr.Markdown('### 📊 Guest History & Extras')
            is_repeated_guest    = gr.Checkbox(value=False,
                                               label='Repeated Guest')
            prev_cancellations   = gr.Slider(0, 26, value=0, step=1,
                                             label='Previous Cancellations')
            prev_not_canceled    = gr.Slider(0, 72, value=0, step=1,
                                             label='Previous Bookings Honored')
            booking_changes      = gr.Slider(0, 21, value=0, step=1,
                                             label='Booking Changes')
            days_in_waiting      = gr.Slider(0, 400, value=0, step=1,
                                             label='Days in Waiting List')
            total_special_reqs   = gr.Slider(0, 5,   value=0, step=1,
                                             label='Total Special Requests')

        with gr.Column(scale=2):
            gr.Markdown('### 🎯 Prediction Result')
            result_box = gr.Textbox(
                label='Risk Assessment',
                lines=24,
                interactive=False,
                elem_id='result-box',
                placeholder='Fill in the booking details on the left, then click PREDICT…'
            )

    # ── Predict button ───────────────────────────────────────────────────────
    predict_btn = gr.Button(
        '🔍  PREDICT CANCELLATION RISK',
        variant='primary',
        elem_id='predict-btn'
    )

    all_inputs = [
        hotel, lead_time, arrival_year, arrival_month,
        arrival_week, arrival_day,
        stays_weekend, stays_week,
        adults, children, babies,
        meal, market_segment, distribution_channel,
        is_repeated_guest, prev_cancellations, prev_not_canceled,
        booking_changes, deposit_type,
        days_in_waiting, customer_type,
        adr, required_parking, total_special_reqs
    ]

    predict_btn.click(fn=predict_cancellation,
                      inputs=all_inputs,
                      outputs=result_box)

    # ── Pre-filled examples ──────────────────────────────────────────────────
    gr.HTML("<br>")
    gr.Markdown('### 📌 Quick Example Scenarios')

    gr.Examples(
        label='Click a row to auto-fill all fields, then click PREDICT',
        examples=[
            # HIGH risk: non-refund deposit, long lead, no special requests, prev cancellation
            ['City Hotel',  365, 2025,  8, 32, 10,  0, 2,
              2, 0, 0, 'SC',  'Online TA',   'TA/TO',
              False, 3, 0, 0, 'Non Refund',  0, 'Transient',  55, 0, 0],
            # LOW risk: short lead, repeated guest, several special requests, no prev cancel
            ['Resort Hotel', 14, 2025,  7, 28, 15,  2, 5,
              2, 1, 0, 'BB', 'Direct',       'Direct',
              True,  0, 4, 1, 'No Deposit',  0, 'Transient', 185, 1, 3],
            # MEDIUM risk: corporate, moderate lead, small deposit
            ['City Hotel',   60, 2025,  9, 38,  5,  0, 3,
              1, 0, 0, 'BB', 'Corporate',    'Corporate',
              False, 0, 1, 0, 'No Deposit',  5, 'Transient', 120, 0, 1],
        ],
        inputs=all_inputs,
        outputs=result_box,
        fn=predict_cancellation,
        cache_examples=False
    )

    # Footer
    gr.HTML("""
    <hr style="border:0; border-top:1px solid #2c5364; margin:28px 0 10px;">
    <p style="text-align:center; color:#78909c; font-size:0.88em;">
        🏨 Hotel Cancellation Predictor
        &nbsp;|&nbsp; Stacking Ensemble: XGBoost · LightGBM · HistGB · Extra Trees · RF
        &nbsp;|&nbsp; 119K samples · 60+ features
        &nbsp;|&nbsp; <b style='color:#ffd200;'>Accuracy 92.93% · AUC 0.9766</b>
    </p>
    """)

# ── Launch ────────────────────────────────────────────────────────────────────
demo.launch(share=False, inbrowser=True)
# Set share=True to get a public link (useful for sharing outside the notebook)